# 从张量形状走进 CLIP：双编码器、对比学习与零样本分类

这是一份**完全离线、CPU 可运行、自包含**的中文教学 Notebook。我们只用 PyTorch 搭一个很小的 CLIP，沿前向传播逐站观察张量，并真的计算损失、反向传播和测试关键性质。

> **重要边界**：本 Notebook 的模型是随机初始化的玩具模型，只能帮助理解结构与数学。它没有学过图文对应关系，**不能进行有意义的零样本识别**。真实 CLIP 的能力来自大规模图文对预训练，而不是架构本身。

## 学习路线

1. 把图像切成 patch token，把文本变成 token 序列；
2. 分别用图像 Transformer 与文本 Transformer 编码；
3. 用图像的 \`[CLS]\` 和文本的 EOS 位置得到全局表征；
4. 投影到同一个空间并做 L2 归一化；
5. 组成温度缩放的图文相似度矩阵；
6. 用对称 InfoNCE（两次交叉熵）训练；
7. 把类别名写成 prompt，理解零样本分类推理。


In [7]:
import math
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)
device = torch.device("cpu")
print("PyTorch:", torch.__version__)
print("device:", device)


PyTorch: 2.11.0+cu128
device: cpu


## 1. 一张图：CLIP 的信息流

CLIP（Contrastive Language–Image Pre-training）是一个**双编码器**：图像和文本不在每层互相注意，而是各自被编码成一个向量，最后才在共享空间计算相似度。这样可以预先缓存一侧向量，检索和零样本分类都很高效。

\`\`\`text
图像 [B,C,H,W] -> patch tokens -> 图像 Transformer -> CLS -> 图像投影 --┐
                                                                    ├-> 归一化 -> 相似度 [B,B]
文本 [B,L]     -> token embedding -> 文本 Transformer -> EOS -> 文本投影 --┘
\`\`\`

设归一化后的图像、文本向量为 $v_i,t_j\in\mathbb{R}^d$，批大小为 $B$。每一行问“这张图最匹配哪段文字”，每一列问“这段文字最匹配哪张图”。训练数据假设第 $i$ 张图和第 $i$ 段文字配对。


## 2. 图像分支：ViT patch token

把 $H\times W$ 图像按 $P\times P$ 不重叠小块切分，共有 $N=(H/P)(W/P)$ 个 patch。\`Conv2d(kernel_size=P, stride=P)\` 可以同时完成切块与线性投影：

$$[B,C,H,W]\rightarrow[B,D,H/P,W/P]\rightarrow[B,N,D].$$

在序列前加入可学习的 \`[CLS]\` token，再加位置嵌入，经 Transformer 后取 \`[CLS]\` 作为整图表示。这里用 pre-norm 残差块；注意力让每个 token 汇总其它位置的信息。


In [8]:
class TransformerBlock(nn.Module):
    def __init__(self, width, heads, mlp_ratio=2):
        super().__init__()
        self.ln1 = nn.LayerNorm(width)
        self.attn = nn.MultiheadAttention(width, heads, batch_first=True)
        self.ln2 = nn.LayerNorm(width)
        self.mlp = nn.Sequential(
            nn.Linear(width, width * mlp_ratio), nn.GELU(),
            nn.Linear(width * mlp_ratio, width),
        )

    def forward(self, x, attn_mask=None):
        h = self.ln1(x)
        x = x + self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)[0]
        return x + self.mlp(self.ln2(x))


class TinyVisionEncoder(nn.Module):
    def __init__(self, image_size=16, patch_size=4, width=32, depth=2, heads=4):
        super().__init__()
        assert image_size % patch_size == 0
        self.patch_embed = nn.Conv2d(3, width, patch_size, stride=patch_size)
        n_patches = (image_size // patch_size) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, width))
        self.pos = nn.Parameter(torch.randn(1, n_patches + 1, width) * 0.01)
        self.blocks = nn.ModuleList([TransformerBlock(width, heads) for _ in range(depth)])
        self.ln = nn.LayerNorm(width)

    def forward(self, images, return_steps=False):
        patch_grid = self.patch_embed(images)
        patch_tokens = patch_grid.flatten(2).transpose(1, 2)
        cls = self.cls.expand(images.shape[0], -1, -1)
        tokens = torch.cat([cls, patch_tokens], dim=1) + self.pos
        for block in self.blocks:
            tokens = block(tokens)
        pooled = self.ln(tokens)[:, 0]
        if return_steps:
            return pooled, {"patch_grid": patch_grid, "patch_tokens": patch_tokens, "with_cls": tokens}
        return pooled


In [9]:
B = 4
images = torch.randn(B, 3, 16, 16, device=device)
vision_demo = TinyVisionEncoder().to(device)
image_pooled, image_steps = vision_demo(images, return_steps=True)
print("输入图像:       ", tuple(images.shape))
print("卷积 patch 网格:", tuple(image_steps["patch_grid"].shape))
print("patch tokens:   ", tuple(image_steps["patch_tokens"].shape))
print("加入 CLS 后:     ", tuple(image_steps["with_cls"].shape))
print("CLS 池化:        ", tuple(image_pooled.shape))
assert image_steps["patch_tokens"].shape == (B, 16, 32)
assert image_pooled.shape == (B, 32)


输入图像:        (4, 3, 16, 16)
卷积 patch 网格: (4, 32, 4, 4)
patch tokens:    (4, 16, 32)
加入 CLS 后:      (4, 17, 32)
CLS 池化:         (4, 32)


## 3. 文本分支：token Transformer 与 EOS 池化

文本先被 tokenizer 转成整数 ID。真实 CLIP 常使用 BPE 等子词 tokenizer；这里直接构造 ID，不依赖任何外部词表。\`0\` 作为 PAD，\`1\` 作为 BOS，\`2\` 作为 EOS。

文本 Transformer 使用**因果注意力掩码**：位置 $i$ 只能看位置 $\le i$ 的 token。最终从每个样本的 EOS 所在位置取向量。不要把“EOS 一定是最大 token ID”当成普遍规则；这里显式查找 EOS，更清楚也更稳健。PAD 的输出虽然仍被计算，但 EOS 位于 PAD 之前且因果掩码不允许它看未来 PAD，因此 EOS 表征不受尾部 padding 影响。


In [10]:
class TinyTextEncoder(nn.Module):
    def __init__(self, vocab_size=64, context_length=8, width=32, depth=2, heads=4, eos_id=2):
        super().__init__()
        self.context_length = context_length
        self.eos_id = eos_id
        self.token_embed = nn.Embedding(vocab_size, width, padding_idx=0)
        self.pos = nn.Parameter(torch.randn(1, context_length, width) * 0.01)
        self.blocks = nn.ModuleList([TransformerBlock(width, heads) for _ in range(depth)])
        self.ln = nn.LayerNorm(width)

    def forward(self, token_ids, return_steps=False):
        B, L = token_ids.shape
        assert L <= self.context_length
        x = self.token_embed(token_ids) + self.pos[:, :L]
        causal_mask = torch.full((L, L), float("-inf"), device=x.device)
        causal_mask = torch.triu(causal_mask, diagonal=1)
        for block in self.blocks:
            x = block(x, attn_mask=causal_mask)
        x = self.ln(x)
        is_eos = token_ids.eq(self.eos_id)
        if not bool(is_eos.any(dim=1).all()):
            raise ValueError("每个文本序列都必须含有 EOS token")
        eos_pos = is_eos.to(torch.int64).argmax(dim=1)
        pooled = x[torch.arange(B, device=x.device), eos_pos]
        if return_steps:
            return pooled, {"embedded": x, "causal_mask": causal_mask, "eos_pos": eos_pos}
        return pooled


In [11]:
token_ids = torch.tensor([
    [1, 11, 12, 2, 0, 0, 0, 0],
    [1, 21, 22, 23, 2, 0, 0, 0],
    [1, 31, 2, 0, 0, 0, 0, 0],
    [1, 41, 42, 43, 44, 2, 0, 0],
], dtype=torch.long, device=device)
text_demo = TinyTextEncoder().to(device)
text_pooled, text_steps = text_demo(token_ids, return_steps=True)
print("token IDs:       ", tuple(token_ids.shape))
print("Transformer 输出:", tuple(text_steps["embedded"].shape))
print("EOS 位置:         ", text_steps["eos_pos"].tolist())
print("EOS 池化:         ", tuple(text_pooled.shape))
print("因果 mask（0 表示可见，-inf 表示不可见）:\n", text_steps["causal_mask"])
assert text_pooled.shape == (B, 32)
assert text_steps["eos_pos"].tolist() == [3, 4, 2, 5]


token IDs:        (4, 8)
Transformer 输出: (4, 8, 32)
EOS 位置:          [3, 4, 2, 5]
EOS 池化:          (4, 32)
因果 mask（0 表示可见，-inf 表示不可见）:
 tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


## 4. 投影、归一化与温度缩放相似度矩阵

两个编码器的原始宽度可以不同。各自的线性投影把它们映射到相同的 $d$ 维共享空间：

$$\tilde v_i=W_v h_i^{(image)},\qquad \tilde t_j=W_t h_j^{(text)}.$$

随后做 L2 归一化，$v_i=\tilde v_i/\|\tilde v_i\|_2$、$t_j=\tilde t_j/\|\tilde t_j\|_2$，点积就成为余弦相似度。logits 为

$$S_{ij}=\exp(s)\,v_i^\top t_j=\frac{v_i^\top t_j}{\tau},$$

其中 $s=\log(1/\tau)$ 是可学习的 logit scale。温度 $\tau$ 越小，softmax 越尖锐。实践中通常会限制 scale，避免数值过大。


In [12]:
class TinyCLIP(nn.Module):
    def __init__(self, embed_dim=24):
        super().__init__()
        self.vision = TinyVisionEncoder()
        self.text = TinyTextEncoder()
        self.image_projection = nn.Linear(32, embed_dim, bias=False)
        self.text_projection = nn.Linear(32, embed_dim, bias=False)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode_image(self, images):
        z = self.image_projection(self.vision(images))
        return F.normalize(z, dim=-1)

    def encode_text(self, token_ids):
        z = self.text_projection(self.text(token_ids))
        return F.normalize(z, dim=-1)

    def forward(self, images, token_ids):
        image_features = self.encode_image(images)
        text_features = self.encode_text(token_ids)
        scale = self.logit_scale.exp().clamp(max=100.0)
        logits_per_image = scale * image_features @ text_features.T
        return logits_per_image, image_features, text_features

model = TinyCLIP().to(device)
logits, image_features, text_features = model(images, token_ids)
print("图像共享向量:", tuple(image_features.shape))
print("文本共享向量:", tuple(text_features.shape))
print("logit scale: ", float(model.logit_scale.exp().detach()))
print("相似度矩阵:   ", tuple(logits.shape))
print(logits.detach())
print("图像向量范数:", image_features.norm(dim=-1).detach())
assert logits.shape == (B, B)
assert torch.allclose(image_features.norm(dim=-1), torch.ones(B), atol=1e-5)
assert torch.allclose(text_features.norm(dim=-1), torch.ones(B), atol=1e-5)


图像共享向量: (4, 24)
文本共享向量: (4, 24)
logit scale:  14.285714149475098
相似度矩阵:    (4, 4)
tensor([[-2.826, -2.851, -2.704, -2.498],
        [-2.714, -2.453, -2.387, -2.291],
        [-3.197, -2.763, -4.184, -3.765],
        [-3.845, -3.859, -4.297, -4.098]])
图像向量范数: tensor([1.000, 1.000, 1.000, 1.000])


## 5. 对称 InfoNCE：为什么是两次交叉熵

一个 batch 内，其它非配对样本充当负样本。标签是 $y_i=i$。图像到文本方向按矩阵每一行分类：

$$\mathcal L_{i\to t}=-\frac1B\sum_{i=1}^B\log\frac{\exp(S_{ii})}{\sum_{j=1}^B\exp(S_{ij})}.$$

文本到图像方向等价于转置矩阵后按行分类：

$$\mathcal L_{t\to i}=-\frac1B\sum_{j=1}^B\log\frac{\exp(S_{jj})}{\sum_{i=1}^B\exp(S_{ij})}.$$

总损失为 $\mathcal L=\frac12(\mathcal L_{i\to t}+\mathcal L_{t\to i})$。它同时要求“图找文”和“文找图”正确。batch 越大，通常可提供越丰富的负样本，但也提高显存和通信成本。语义相同却未标为配对的样本可能成为假负样本。


In [13]:
def symmetric_clip_loss(logits):
    labels = torch.arange(logits.shape[0], device=logits.device)
    loss_i2t = F.cross_entropy(logits, labels)
    loss_t2i = F.cross_entropy(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2, loss_i2t, loss_t2i

model.zero_grad(set_to_none=True)
logits, _, _ = model(images, token_ids)
loss, loss_i2t, loss_t2i = symmetric_clip_loss(logits)
print(f"image -> text CE: {loss_i2t.item():.4f}")
print(f"text -> image CE: {loss_t2i.item():.4f}")
print(f"symmetric loss:   {loss.item():.4f}")
loss.backward()

checks = {
    "vision patch_embed": model.vision.patch_embed.weight.grad,
    "text token_embed": model.text.token_embed.weight.grad,
    "image projection": model.image_projection.weight.grad,
    "text projection": model.text_projection.weight.grad,
    "logit scale": model.logit_scale.grad,
}
for name, grad in checks.items():
    norm = None if grad is None else grad.norm().item()
    print(f"{name:20s} grad norm = {norm}")
    assert grad is not None and torch.isfinite(grad).all() and grad.norm() > 0
assert torch.isfinite(loss)


image -> text CE: 1.6510
text -> image CE: 1.8095
symmetric loss:   1.7302
vision patch_embed   grad norm = 6.3702569007873535
text token_embed     grad norm = 0.36961349844932556
image projection     grad norm = 4.234209060668945
text projection      grad norm = 4.059686660766602
logit scale          grad norm = 0.44986578822135925


## 6. 零样本分类：类别名变成文本分类器

预训练后，不再需要训练一个固定的线性分类头。对每个类别构造 prompt，例如“**一张猫的照片**”“**一张狗的照片**”“**一张汽车的照片**”。

把 $K$ 个 prompt 编码成文本向量 $T\in\mathbb R^{K\times d}$；图像向量 $v$ 与所有类别文本做相似度：

$$p(y=k\mid x)=\operatorname{softmax}(\exp(s)vT^\top)_k.$$

真实应用常用多种 prompt 模板分别编码，再对同一类的归一化向量求平均并再次归一化，以减少措辞敏感性。下面用三段**伪 token**演示接口。输出只是随机数，不代表模型认出了图像。


In [14]:
class_names = ["猫", "狗", "汽车"]
prompt_token_ids = torch.tensor([
    [1, 10, 50, 2, 0, 0, 0, 0],
    [1, 10, 51, 2, 0, 0, 0, 0],
    [1, 10, 52, 2, 0, 0, 0, 0],
], dtype=torch.long, device=device)

model.eval()
with torch.no_grad():
    one_image = model.encode_image(images[:1])
    class_text = model.encode_text(prompt_token_ids)
    zero_shot_logits = model.logit_scale.exp().clamp(max=100) * one_image @ class_text.T
    probabilities = zero_shot_logits.softmax(dim=-1)

print("类别:", class_names)
print("随机权重产生的“概率”（仅检查流程）:", probabilities.squeeze(0))
print("概率和:", probabilities.sum().item())
assert probabilities.shape == (1, 3)
assert torch.allclose(probabilities.sum(dim=-1), torch.ones(1), atol=1e-6)


类别: ['猫', '狗', '汽车']
随机权重产生的“概率”（仅检查流程）: tensor([0.479, 0.233, 0.288])
概率和: 1.0


## 7. 训练与推理有什么不同？

| 阶段 | 输入 | 核心计算 | 更新参数 | 输出用途 |
|---|---|---|---|---|
| 对比预训练 | 成对图像与文本，通常大 batch | $B\times B$ 相似度 + 对称 InfoNCE | 两个编码器、投影、温度 | 学到可迁移的共享空间 |
| 零样本推理 | 图像 + 类别 prompts | 图像对 $K$ 个类别文本的相似度 | 否，\`eval()\` + \`no_grad()\` | 分类概率或排序 |
| 图文检索 | 查询来自一侧，候选来自另一侧 | 缓存候选向量后做最近邻 | 否 | top-k 候选 |

推理时可以缓存类别文本向量；换一组类别只需重新编码文本。\`eval()\` 会关闭 dropout 等训练行为，\`torch.no_grad()\` 减少内存，但它们都不会让随机权重突然获得语义。

## 8. 局限性与使用边界

- **数据偏差与社会偏见**：模型会继承网页图文数据里的刻板印象、不平衡和有害关联。
- **prompt 敏感**：类别名和模板措辞会显著改变结果，不同语言表现可能不均衡。
- **细粒度与关系理解**：外观接近的类别、计数、否定、空间关系和组合推理可能困难。
- **分布外与校准**：相似度高不等于事实正确，softmax 也不保证可靠置信度。
- **批内负样本假设**：语义相近的未配对样本会被当成负样本；噪声也会污染监督。
- **双编码器的代价**：高效但缺少逐 token 的跨模态交互，复杂对齐往往不如 cross-encoder。
- **安全与治理**：部署需做领域评估、偏差审计、阈值设计和人工复核，不能把分数直接当可靠判断。


## 9. 单元测试：把关键理解写成断言

测试不追求“随机模型准确”，而验证结构不变量：形状正确、归一化正确、对角占优时损失更低、对称损失对交换模态不变、梯度可达关键参数。


In [15]:
def run_unit_tests():
    test_model = TinyCLIP(embed_dim=12).to(device)
    test_images = torch.randn(3, 3, 16, 16, device=device)
    test_tokens = torch.tensor([
        [1, 5, 2, 0, 0, 0, 0, 0],
        [1, 6, 7, 2, 0, 0, 0, 0],
        [1, 8, 9, 10, 2, 0, 0, 0],
    ], device=device)
    test_logits, vi, tx = test_model(test_images, test_tokens)
    assert vi.shape == tx.shape == (3, 12)
    assert test_logits.shape == (3, 3)
    assert torch.allclose(vi.norm(dim=-1), torch.ones(3), atol=1e-5)
    assert torch.allclose(tx.norm(dim=-1), torch.ones(3), atol=1e-5)

    good = torch.eye(4) * 10.0
    flat = torch.zeros(4, 4)
    assert symmetric_clip_loss(good)[0] < symmetric_clip_loss(flat)[0]

    random_logits = torch.randn(4, 4)
    assert torch.allclose(
        symmetric_clip_loss(random_logits)[0],
        symmetric_clip_loss(random_logits.T)[0], atol=1e-7
    )

    try:
        test_model.text(torch.tensor([[1, 5, 0, 0, 0, 0, 0, 0]], device=device))
        raise AssertionError("缺少 EOS 时本应报错")
    except ValueError:
        pass

    test_model.zero_grad(set_to_none=True)
    symmetric_clip_loss(test_logits)[0].backward()
    required = [test_model.vision.cls, test_model.text.pos,
                test_model.image_projection.weight, test_model.logit_scale]
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in required)
    print("全部单元测试通过（5/5）")

run_unit_tests()


全部单元测试通过（5/5）


## 10. 思考题

1. 若不做 L2 归一化，模型能否仅通过增大向量范数降低损失？这会怎样影响“相似度”的解释？
2. 为什么损失要同时计算图到文和文到图？只用一个方向会遗漏什么约束？
3. batch size 从 4 增到 4096 时，负样本数量、计算量和假负样本概率分别如何变化？
4. 温度趋近 0 时 softmax 与梯度会怎样？为什么要学习温度并限制最大 scale？
5. 将文本 EOS 池化替换成平均池化，需要怎样正确排除 PAD？两者各有什么归纳偏置？
6. 为什么 prompt ensemble 可能比单模板稳定？中文类别名、多义词和领域术语会带来什么问题？
7. 双编码器适合大规模检索，而 cross-encoder 往往更精细：怎样设计两阶段系统兼顾速度和精度？
8. 设计一个实验，区分“模型确实理解空间关系”与“只利用数据集捷径”。

### 小结

CLIP 的核心可以浓缩为：**独立编码图像和文本 → 投影并归一化 → 温度缩放的全配对相似度 → 双向交叉熵**。零样本分类只是把类别写成文本，让预训练形成的共享空间充当动态分类器。架构给出机制，语义能力来自数据和训练；因此这个玩具实现能验证计算图，却不能替代预训练模型。
